<a href="https://colab.research.google.com/github/NU-MSE-LECTURES/465-WINTER2026/blob/main/Week_07/lectures/lecture_7.3_strain_field_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 7.3: Quantitative Strain and Electric Field Mapping

## Overview

This lecture demonstrates advanced **quantitative analysis** of 4D-STEM data to map **strain tensors** and **electric fields** at the nanoscale. We'll combine techniques from previous lectures and add new methods for measuring lattice distortions and their physical origins.

## Physical Background

### Strain in Crystalline Materials

**Strain** describes local lattice distortion relative to a reference state. The strain tensor $\varepsilon_{ij}$ relates the deformed lattice to the undeformed lattice:

$$
\varepsilon_{ij} = \frac{1}{2}\left(\frac{\partial u_i}{\partial x_j} + \frac{\partial u_j}{\partial x_i}\right)
$$

where $\vec{u}$ is the displacement field.

In 2D (thin film):
$$
\boldsymbol{\varepsilon} = \begin{bmatrix} \varepsilon_{xx} & \varepsilon_{xy} \\ \varepsilon_{xy} & \varepsilon_{yy} \end{bmatrix}
$$

### Sources of Strain

- **Elastic deformation**: stress, bending
- **Composition gradients**: alloys, dopants
- **Defects**: dislocations, grain boundaries
- **Heterostructures**: lattice mismatch
- **Thermal expansion**: temperature gradients

### Electric Fields

Electric fields in materials arise from:
- **Space charge**: dopants, charged defects
- **Interfaces**: Schottky barriers, p-n junctions
- **Polarization**: piezoelectric/ferroelectric materials
- **Built-in potentials**: work function differences

## Learning Objectives

By the end of this lecture, you will:
1. Measure lattice vectors from diffraction patterns
2. Calculate strain tensor components from lattice distortions
3. Map strain fields with nanometer resolution
4. Quantify electric fields using DPC
5. Correlate strain and electric field distributions
6. Estimate measurement uncertainties
7. Interpret physical origins of observed fields

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse, Circle
import h5py
from scipy import ndimage
from scipy.optimize import curve_fit
from scipy.fft import fft2, ifft2, fftshift, ifftshift
import os

print(f"NumPy version: {np.__version__}")

# Try to import py4DSTEM
try:
    import py4DSTEM
    has_py4dstem = True
    print(f"py4DSTEM version: {py4DSTEM.__version__}")
except ImportError:
    has_py4dstem = False
    print("py4DSTEM not available - using manual implementation")

## Part 1: Loading Data and Previous Results

We'll build on the COM analysis from Lecture 7.2 and add strain measurements.

In [ ]:
# Load 4D-STEM datacube
data_dir = "/Users/robertoreis/Documents/codes/465_Computational_Microscopy_2026/data_4dstem/SmB6"
master_file = os.path.join(data_dir, "SmB6_10us_1kpx_HM_4_master.h5")

print("Loading 4D-STEM data...")

# Load with stride for memory management
try:
    if has_py4dstem:
        datacube = py4DSTEM.import_file(master_file)
        if hasattr(datacube, 'data'):
            datacube_array = datacube.data
        else:
            datacube_array = np.array(datacube)
    else:
        raise Exception("Using manual loading")
except:
    with h5py.File(master_file, 'r') as f:
        # Find data path
        data_path = '/entry/data/data'
        # You may need to adjust this path based on file structure
        
        stride = 2  # Subsample for memory
        data = f[data_path]
        datacube_array = data[::stride, ::stride, :, :][:]

Rx, Ry, Qx, Qy = datacube_array.shape
print(f"Loaded datacube: ({Rx}, {Ry}, {Qx}, {Qy})")

# Calibration (from Lecture 7.1)
calibration = {
    'R_pixel_size_nm': 1.0,  # Real space scan step
    'Q_pixel_size_mrad': 1.0,  # Angular pixel size
    'Q_pixel_size_invA': 0.025,  # Reciprocal space calibration (1/Å)
    'beam_center': (Qx // 2, Qy // 2),
    'energy_keV': 200.0,
    'wavelength_pm': 2.51,
}

print(f"Calibration set: {calibration['Q_pixel_size_invA']:.4f} Å⁻¹/pixel")

## Part 2: Bragg Disk Detection

To measure strain, we need to locate **Bragg peaks** (diffraction spots) in each pattern.

### Strategy:
1. **Identify peaks** in mean diffraction pattern
2. **Track peaks** across all scan positions
3. **Measure shifts** relative to reference positions
4. **Calculate strain** from shift gradients

### Peak Finding Methods:

**Template Matching:**
- Cross-correlate disk template with each pattern
- Find maximum correlation → disk center

**Peak Fitting:**
- Fit 2D Gaussian to each peak
- Extract center position with sub-pixel precision

**py4DSTEM Disk Detection:**
- Find Bragg disks in mean/max DP
- Cross-correlate to find in each position

In [ ]:
# Compute mean diffraction pattern and identify Bragg peaks

mean_dp = np.mean(datacube_array, axis=(0, 1))
max_dp = np.max(datacube_array, axis=(0, 1))

print("Finding Bragg peaks in mean diffraction pattern...")

# Simplified peak finding: locate local maxima
# For real analysis, use more sophisticated methods (py4DSTEM, template matching)

# Apply Gaussian blur to reduce noise
from scipy.ndimage import gaussian_filter, maximum_filter

mean_dp_smooth = gaussian_filter(mean_dp, sigma=2)

# Find local maxima
local_max = maximum_filter(mean_dp_smooth, size=10)
peaks_mask = (mean_dp_smooth == local_max) & (mean_dp_smooth > np.percentile(mean_dp_smooth, 99))

# Get peak positions
peak_positions = np.argwhere(peaks_mask)
print(f"Found {len(peak_positions)} candidate peaks")

# Remove peaks too close to center (direct beam) or edges
center_x, center_y = calibration['beam_center']
filtered_peaks = []

for py, px in peak_positions:
    dist_from_center = np.sqrt((px - center_x)**2 + (py - center_y)**2)
    if 15 < dist_from_center < min(Qx, Qy)//2 - 10:  # Exclude direct beam and edges
        filtered_peaks.append([py, px])

filtered_peaks = np.array(filtered_peaks)
print(f"After filtering: {len(filtered_peaks)} Bragg peaks")

# Select strongest peaks (e.g., top 6-12 for common crystal structures)
peak_intensities = [mean_dp[py, px] for py, px in filtered_peaks]
sorted_indices = np.argsort(peak_intensities)[::-1]
num_peaks_to_use = min(12, len(filtered_peaks))
selected_peaks = filtered_peaks[sorted_indices[:num_peaks_to_use]]

print(f"Selected {len(selected_peaks)} strongest peaks for analysis")

# Visualize peaks
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.imshow(np.log(mean_dp + 1), cmap='gray', interpolation='nearest')
ax.scatter(selected_peaks[:, 1], selected_peaks[:, 0], 
          c='red', s=200, facecolors='none', linewidths=2, label='Selected peaks')
ax.scatter([center_x], [center_y], c='cyan', s=100, marker='+', 
          linewidths=3, label='Direct beam')
ax.set_title('Mean Diffraction Pattern\nwith Identified Bragg Peaks', 
            fontsize=12, weight='bold')
ax.set_xlabel('Qx (pixels)')
ax.set_ylabel('Qy (pixels)')
ax.legend(framealpha=0.9)
plt.colorbar(im, ax=ax, label='log(Intensity)')

# Show zoomed region around a peak
if len(selected_peaks) > 0:
    peak_idx = 0
    py, px = selected_peaks[peak_idx]
    zoom_size = 20
    
    y_slice = slice(max(0, py-zoom_size), min(Qy, py+zoom_size))
    x_slice = slice(max(0, px-zoom_size), min(Qx, px+zoom_size))
    
    ax = axes[1]
    zoomed = mean_dp[y_slice, x_slice]
    im = ax.imshow(zoomed, cmap='hot', interpolation='nearest')
    ax.set_title(f'Zoom: Peak at ({px}, {py})', fontsize=12, weight='bold')
    ax.axhline(zoom_size, color='cyan', linestyle='--', linewidth=1)
    ax.axvline(zoom_size, color='cyan', linestyle='--', linewidth=1)
    plt.colorbar(im, ax=ax, label='Intensity')

plt.tight_layout()
plt.show()

## Part 3: Strain Tensor Measurement

### Theory

From measured Bragg disk positions, we can determine **strain** by comparing local lattice vectors to a reference.

**Lattice vectors** in reciprocal space:
$$
\vec{g}_1, \vec{g}_2, \vec{g}_3, \ldots
$$

**Strain components** from displacements $\delta\vec{g}_i$:
$$
\varepsilon_{xx} = \frac{\delta g_x}{g_{x,0}}
$$
$$
\varepsilon_{yy} = \frac{\delta g_y}{g_{y,0}}
$$
$$
\varepsilon_{xy} = \frac{1}{2}\left(\frac{\delta g_x}{g_{y,0}} + \frac{\delta g_y}{g_{x,0}}\right)
$$

### Measurement Strategy

**For each scan position:**
1. Locate Bragg disks (cross-correlation or template matching)
2. Measure position shifts relative to reference
3. Calculate strain from multiple $\vec{g}$-vectors
4. Average over equivalent reflections for precision

### Reference Selection

**Reference lattice** $\vec{g}_0$ options:
- **Global reference**: mean over entire scan
- **Local reference**: running average (for gradients)
- **Vacuum region**: if available
- **Known standard**: calibration sample

In [ ]:
def track_peak_shifts(datacube, peak_position, beam_center, search_radius=15):
    """
    Track a Bragg peak across all scan positions using cross-correlation.
    
    Parameters:
    -----------
    datacube : ndarray (Rx, Ry, Qx, Qy)
    peak_position : tuple (py, px)
        Peak location in mean DP
    beam_center : tuple (cx, cy)
    search_radius : int
        Half-size of search window
    
    Returns:
    --------
    shifts_x, shifts_y : ndarrays (Rx, Ry)
        Peak shifts relative to reference position
    """
    Rx, Ry, Qx, Qy = datacube.shape
    py, px = peak_position
    
    # Create template from mean DP
    template_size = search_radius
    y_slice = slice(max(0, py-template_size), min(Qy, py+template_size))
    x_slice = slice(max(0, px-template_size), min(Qx, px+template_size))
    template = np.mean(datacube[:, :, y_slice, x_slice], axis=(0, 1))
    template = (template - np.mean(template)) / np.std(template)  # Normalize
    
    shifts_x = np.zeros((Rx, Ry))
    shifts_y = np.zeros((Rx, Ry))
    
    print(f"Tracking peak at ({px}, {py})...")
    
    for i in range(Rx):
        if i % max(1, Rx//10) == 0:
            print(f"  Progress: {100*i//Rx}%")
        
        for j in range(Ry):
            # Extract search region
            search_window = datacube[i, j, y_slice, x_slice]
            search_window = (search_window - np.mean(search_window)) / (np.std(search_window) + 1e-10)
            
            # Cross-correlation
            from scipy.signal import correlate2d
            corr = correlate2d(search_window, template, mode='same')
            
            # Find peak of correlation
            max_pos = np.unravel_index(np.argmax(corr), corr.shape)
            
            # Calculate shift from center
            center_y, center_x = np.array(corr.shape) // 2
            shifts_y[i, j] = max_pos[0] - center_y
            shifts_x[i, j] = max_pos[1] - center_x
    
    print("  Progress: 100% - Complete!")
    
    return shifts_x, shifts_y

# Track the first few strong peaks
# (For full analysis, track all selected peaks)

num_peaks_to_track = min(4, len(selected_peaks))  # Limit for demonstration
peak_shifts_x = []
peak_shifts_y = []

for idx in range(num_peaks_to_track):
    peak_pos = selected_peaks[idx]
    shifts_x, shifts_y = track_peak_shifts(
        datacube_array, 
        peak_pos, 
        calibration['beam_center'],
        search_radius=12
    )
    peak_shifts_x.append(shifts_x)
    peak_shifts_y.append(shifts_y)

print(f"\nTracked {num_peaks_to_track} peaks across scan")

# Visualize shifts for first peak
if peak_shifts_x:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    peak_idx = 0
    
    ax = axes[0, 0]
    im = ax.imshow(peak_shifts_x[peak_idx], cmap='RdBu_r', interpolation='nearest')
    ax.set_title(f'Peak {peak_idx+1}: Shift in X', fontsize=11, weight='bold')
    ax.set_xlabel('Scan X')
    ax.set_ylabel('Scan Y')
    plt.colorbar(im, ax=ax, fraction=0.046, label='Δx (pixels)')
    
    ax = axes[0, 1]
    im = ax.imshow(peak_shifts_y[peak_idx], cmap='RdBu_r', interpolation='nearest')
    ax.set_title(f'Peak {peak_idx+1}: Shift in Y', fontsize=11, weight='bold')
    ax.set_xlabel('Scan X')
    ax.set_ylabel('Scan Y')
    plt.colorbar(im, ax=ax, fraction=0.046, label='Δy (pixels)')
    
    ax = axes[1, 0]
    shift_magnitude = np.sqrt(peak_shifts_x[peak_idx]**2 + peak_shifts_y[peak_idx]**2)
    im = ax.imshow(shift_magnitude, cmap='viridis', interpolation='nearest')
    ax.set_title(f'Peak {peak_idx+1}: Shift Magnitude', fontsize=11, weight='bold')
    ax.set_xlabel('Scan X')
    ax.set_ylabel('Scan Y')
    plt.colorbar(im, ax=ax, fraction=0.046, label='|Δ| (pixels)')
    
    ax = axes[1, 1]
    ax.hist(peak_shifts_x[peak_idx].ravel(), bins=30, alpha=0.6, 
           label='Shift X', color='blue', density=True)
    ax.hist(peak_shifts_y[peak_idx].ravel(), bins=30, alpha=0.6, 
           label='Shift Y', color='red', density=True)
    ax.set_xlabel('Shift (pixels)')
    ax.set_ylabel('Density')
    ax.set_title('Distribution of Shifts', fontsize=11, weight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.suptitle(f'Bragg Peak Tracking: Peak at {tuple(selected_peaks[peak_idx])}',
                fontsize=13, weight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\nPeak {peak_idx+1} statistics:")
    print(f"  Mean shift: ({np.mean(peak_shifts_x[peak_idx]):.3f}, {np.mean(peak_shifts_y[peak_idx]):.3f}) pixels")
    print(f"  Std shift: ({np.std(peak_shifts_x[peak_idx]):.3f}, {np.std(peak_shifts_y[peak_idx]):.3f}) pixels")
    print(f"  Max magnitude: {np.max(shift_magnitude):.3f} pixels")

## Part 4: Strain Calculation

Now we calculate strain tensor components from the measured peak shifts.

### Infinitesimal Strain Tensor

$$
\boldsymbol{\varepsilon} = \begin{bmatrix}
\varepsilon_{xx} & \varepsilon_{xy} \\
\varepsilon_{xy} & \varepsilon_{yy}
\end{bmatrix}
$$

where:
- $\varepsilon_{xx}$: normal strain in x
- $\varepsilon_{yy}$: normal strain in y
- $\varepsilon_{xy}$: shear strain

### From Multiple Bragg Peaks

Use **least-squares fitting** to determine strain from all measured peaks:

$$
\begin{bmatrix} \delta g_x^{(i)} \\ \delta g_y^{(i)} \end{bmatrix} =
\begin{bmatrix} \varepsilon_{xx} & \varepsilon_{xy} \\ \varepsilon_{xy} & \varepsilon_{yy} \end{bmatrix}
\begin{bmatrix} g_x^{(i)} \\ g_y^{(i)} \end{bmatrix}
$$

Solve for $\varepsilon_{ij}$ overdetermined system.

### Rotation and Strain Decomposition

The deformation gradient can be decomposed:
$$
\mathbf{F} = \mathbf{R} \cdot \mathbf{U}
$$

where $\mathbf{R}$ is rotation and $\mathbf{U}$ is stretch (related to strain).

**Rotation angle:**
$$
\theta = \frac{1}{2}(\varepsilon_{xy}^{+} - \varepsilon_{xy}^{-})
$$

where superscripts indicate off-diagonal symmetry.

In [ ]:
# Calculate strain from peak shifts
# For demonstration, we'll use simplified single-peak strain

def calculate_strain_single_peak(shifts_x, shifts_y, g_vector, Q_calibration):
    """
    Calculate strain from single Bragg peak shifts.
    
    This is simplified - full analysis uses multiple peaks!
    
    Parameters:
    -----------
    shifts_x, shifts_y : ndarrays (Rx, Ry)
        Peak position shifts
    g_vector : array
        Reference g-vector [gx, gy] in pixels
    Q_calibration : float
        Å⁻¹ per pixel
    
    Returns:
    --------
    strain_xx, strain_yy, strain_xy : ndarrays
        Strain tensor components
    """
    # Convert shifts to fractional strain
    # ε = Δg / g
    
    gx, gy = g_vector
    g_magnitude = np.sqrt(gx**2 + gy**2)
    
    # Normal strain along g-vector direction
    # Project shift onto g direction
    g_unit = np.array([gx, gy]) / g_magnitude
    
    # For simplicity, assume shifts represent lattice distortion
    # True analysis requires multiple g-vectors
    
    # Approximate strain (this is oversimplified!)
    strain_xx = shifts_x / gx if gx != 0 else np.zeros_like(shifts_x)
    strain_yy = shifts_y / gy if gy != 0 else np.zeros_like(shifts_y)
    strain_xy = 0.5 * (shifts_x / gy + shifts_y / gx) if (gx != 0 and gy != 0) else np.zeros_like(shifts_x)
    
    # Remove mean (reference to average lattice)
    strain_xx -= np.mean(strain_xx)
    strain_yy -= np.mean(strain_yy)
    strain_xy -= np.mean(strain_xy)
    
    return strain_xx, strain_yy, strain_xy

# Calculate strain from first tracked peak
if peak_shifts_x:
    peak_idx = 0
    py, px = selected_peaks[peak_idx]
    cx, cy = calibration['beam_center']
    
    g_vector = [px - cx, py - cy]  # g-vector in pixels
    
    print(f"\nCalculating strain from peak at {tuple(selected_peaks[peak_idx])}")
    print(f"  g-vector: ({g_vector[0]}, {g_vector[1]}) pixels")
    print(f"  |g|: {np.sqrt(g_vector[0]**2 + g_vector[1]**2):.1f} pixels")
    
    strain_xx, strain_yy, strain_xy = calculate_strain_single_peak(
        peak_shifts_x[peak_idx],
        peak_shifts_y[peak_idx],
        g_vector,
        calibration['Q_pixel_size_invA']
    )
    
    # Also calculate rotation and dilatation
    rotation = strain_xy  # Simplified
    dilatation = strain_xx + strain_yy  # Trace of strain tensor
    
    print(f"\nStrain statistics:")
    print(f"  ε_xx: {np.mean(strain_xx):.6f} ± {np.std(strain_xx):.6f}")
    print(f"  ε_yy: {np.mean(strain_yy):.6f} ± {np.std(strain_yy):.6f}")
    print(f"  ε_xy: {np.mean(strain_xy):.6f} ± {np.std(strain_xy):.6f}")
    print(f"  Dilatation: {np.mean(dilatation):.6f} ± {np.std(dilatation):.6f}")
    
    # Visualize strain tensor
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    
    # Strain components
    components = [
        (strain_xx, 'ε$_{xx}$', 'bwr'),
        (strain_yy, 'ε$_{yy}$', 'bwr'),
        (strain_xy, 'ε$_{xy}$', 'PRGn'),
    ]
    
    for idx, (data, label, cmap) in enumerate(components):
        ax = axes[0, idx]
        vmax = max(abs(np.percentile(data, 1)), abs(np.percentile(data, 99)))
        im = ax.imshow(data, cmap=cmap, interpolation='nearest', 
                      vmin=-vmax, vmax=vmax)
        ax.set_title(label, fontsize=13, weight='bold')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        cbar = plt.colorbar(im, ax=ax, fraction=0.046)
        cbar.set_label('Strain', fontsize=10)
    
    # Derived quantities
    ax = axes[1, 0]
    im = ax.imshow(dilatation, cmap='coolwarm', interpolation='nearest')
    ax.set_title('Dilatation (ε$_{xx}$ + ε$_{yy}$)', fontsize=13, weight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    plt.colorbar(im, ax=ax, fraction=0.046, label='Dilatation')
    
    ax = axes[1, 1]
    im = ax.imshow(rotation, cmap='twilight', interpolation='nearest')
    ax.set_title('Rotation', fontsize=13, weight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    plt.colorbar(im, ax=ax, fraction=0.046, label='Rotation')
    
    # Maximum shear strain
    max_shear = np.sqrt((strain_xx - strain_yy)**2 + 4*strain_xy**2) / 2
    ax = axes[1, 2]
    im = ax.imshow(max_shear, cmap='hot', interpolation='nearest')
    ax.set_title('Maximum Shear Strain', fontsize=13, weight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    plt.colorbar(im, ax=ax, fraction=0.046, label='γ$_{max}$')
    
    plt.suptitle('Strain Tensor Components and Derived Quantities', 
                fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()